In [ ]:
!pip install pypdf chromadb sentence-transformers openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import chromadb
import uuid

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    #old nvdia key i used --->api_key="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" # Replace with your valid OpenRouter API key
    api_key="API_1"
)

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving AI_REPORT.pdf to AI_REPORT.pdf


In [ ]:
def load_pdf(file):

    reader = PdfReader(file)

    pages = []

    for i, page in enumerate(reader.pages):

        text = page.extract_text()

        pages.append({
            "page": i+1,
            "text": text
        })

    return pages

In [ ]:
pages = load_pdf("AI_REPORT.pdf")

print("Pages loaded:", len(pages))

Pages loaded: 17


In [ ]:
def chunk_text(pages, chunk_size=500, overlap=100):

    chunks = []

    for page in pages:

        text = page["text"]

        start = 0

        while start < len(text):

            chunk = text[start:start+chunk_size]

            chunks.append({
                "page": page["page"],
                "text": chunk
            })

            start += chunk_size - overlap

    return chunks

In [ ]:
chunks = chunk_text(pages)

print("Total chunks:", len(chunks))

Total chunks: 104


In [ ]:
def create_embeddings(chunks):

    texts = [c["text"] for c in chunks]

    embeddings = embedding_model.encode(texts)

    return embeddings

In [ ]:
embeddings = create_embeddings(chunks)

print("Embeddings created:", len(embeddings))

Embeddings created: 104


In [ ]:
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="document_store")
except:
    pass

collection = chroma_client.create_collection(name="document_store")

In [ ]:
for i, chunk in enumerate(chunks):

    collection.add(

        ids=[str(uuid.uuid4())],

        documents=[chunk["text"]],

        embeddings=[embeddings[i].tolist()],

        metadatas=[{"page": chunk["page"]}]
    )

print("Chunks stored in ChromaDB")

Chunks stored in ChromaDB


In [ ]:
def retrieve_chunks(query, k=3):

    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )

    return results

In [ ]:
retrieve_chunks("What is artificial intelligence?")

{'ids': [['708e01a0-756e-4f92-b5db-97c1c63f5629',
   '0672ea79-3d36-41d6-91d4-442b714291f0',
   '209bd946-860a-4294-abb7-6b12a20728e6']],
 'embeddings': None,
 'documents': [['Artificial  Intelligence:  Opportunities,  \nApplications\n \nand\n \nConcerns\n \n1.  Introduction  \nArtificial  Intelligence  (AI)  is  one  of  the  most  revolutionary  technological  advancements  of  the  \n21st\n \ncentury.\n \nIt\n \nrefers\n \nto\n \nthe\n \nsimulation\n \nof\n \nhuman\n \nintelligence\n \nin\n \nmachines\n \nthat\n \nare\n \nprogrammed\n \nto\n \nthink,\n \nlearn,\n \nand\n \nperform\n \ntasks\n \nthat\n \nnormally\n \nrequire\n \nhuman\n \ncognitive\n \nabilities.\n \nThese\n \ntasks\n \ninclude\n \nunderstanding\n \nlanguage,\n \nrecogni',
   'ire\n \nhuman\n \ncognitive\n \nabilities.\n \nThese\n \ntasks\n \ninclude\n \nunderstanding\n \nlanguage,\n \nrecognizing\n \npatterns,\n \nsolving\n \nproblems,\n \nmaking\n \ndecisions,\n \nand\n \neven\n \nlearning\n \nfrom\n \nexperience.\

In [ ]:
def call_llm(prompt):

    response = client.chat.completions.create(

       #old nvidia model i used ----> model="nvidia/nemotron-nano-9b-v2:free",
        model="nvidia/nemotron-3-super-120b-a12b:free",

        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
full_document = " ".join([p["text"] for p in pages])

def summarize_tool():

    prompt = f"""
    Summarize the following document:

    {full_document}
    """

    return call_llm(prompt)

In [ ]:
print(summarize_tool())

Here is a concise summary of the key points from the document:

Artificial Intelligence (AI) simulates human intelligence in machines, enabling them to learn, reason, perceive, and make decisions. Its evolution spans from early concepts (like the Turing Test in 1950) and the formal birth of AI at the 1956 Dartmouth Conference, through periods of slowed progress ("AI Winters"), to rapid advancement driven by machine learning, deep learning, big data, and cloud computing since the 2010s.

AI is categorized by **capability** (Narrow/Weak AI for specific tasks, prevalent today; theoretical General/Strong AI for human-level cognition; hypothetical Super AI surpassing humans) and **functionality** (Reactive Machines, Limited Memory AI, theoretical Theory of Mind AI, and theoretical Self-Aware AI). Core technologies enabling AI include machine learning (especially deep learning), natural language processing (NLP), computer vision, robotics, big data, and cloud computing.

AI transforms numero

In [ ]:
def qa_tool(query):

    results = retrieve_chunks(query)

    context = "\n\n".join(results["documents"][0])

    pages = results["metadatas"][0]

    prompt = f"""
    Use the context below to answer the question.

    Context:
    {context}

    Question:
    {query}
    """

    answer = call_llm(prompt)

    citations = [f"Page {m['page']}" for m in pages]

    return answer, citations

In [ ]:
answer, sources = qa_tool("What ethical concerns are mentioned in AI?")

print(answer)
print("Sources:", sources)

The text lists several ethical concerns associated with the development and use of artificial intelligence:

- **Accountability** – Who is responsible when AI systems make decisions or cause harm?  
- **Transparency** – How understandable and open are the processes and decisions of AI systems?  
- **Potential misuse of technology** – The risk that AI could be employed for harmful or unethical purposes.  
- **Fairness and discrimination** – Concerns that AI may produce unfair or discriminatory outcomes.  These are the ethical issues highlighted in the provided context.
Sources: ['Page 16', 'Page 17', 'Page 17']


In [ ]:
def agent_router(query):

    prompt = f"""
    Classify the query.

    If the user asks for summary or overview or brief return "summarize".

    Otherwise return "qa".

    Query: {query}
    """

    decision = call_llm(prompt)

    return decision.lower()

In [ ]:
def ask(query):

    tool = agent_router(query)

    if "summarize" in tool:

        result = summarize_tool()

        print("Tool used: Summarize Tool\n")

        return result

    else:

        answer, sources = qa_tool(query)

        print("Tool used: Q&A Tool\n")

        return answer + "\n\nSources: " + ", ".join(sources)

In [ ]:
while True:

    query = input("\nAsk something about the document (type 'exit' to stop): ")

    if query.lower() == "exit":
        print("Session ended.")
        break

    response = ask(query)

    print("\nAI Response:\n")
    print(response)


Ask something about the document (type 'exit' to stop): what are the applications of AI
Tool used: Summarize Tool


AI Response:

**Summary of the Document on Artificial Intelligence**

**1. What Is AI?**  
AI simulates human intelligence in machines, enabling them to understand language, recognize patterns, solve problems, make decisions, and learn from experience. It excels at processing large datasets and making predictions or recommendations.

**2. Historical Evolution**  
- **1940s‑1950s:** Foundations laid by Alan Turing (Turing Test) and early computers; term “Artificial Intelligence” coined at the 1956 Dartmouth Conference.  - **1960s‑1980s:** Rule‑based systems emerged; progress stalled during the “AI Winter” due to limited computing power and data.  
- **1990s‑2000s:** Renewed interest via machine learning; milestone IBM Deep Blue defeating Kasparov (1997).  
- **2010s‑Present:** Explosion of big data, GPUs, and deep learning; breakthroughs in image/speech recognition, NLP, 

In [ ]:
!pip install gradio
import gradio as gr


In [ ]:
def classify_intent(query):

    prompt = f"""
    You are an intent classifier.

    If the user asks for a summary or overview  or brief of the document,
    respond with only the word:

    summarize

    If the user asks a specific question about the document,
    respond with only the word:

    qa

    User Query: {query}
    """

    decision = call_llm(prompt)

    return decision.strip().lower()

In [ ]:
def ask_question(query):

    intent = agent_router(query)

    if "summarize" in intent:

        answer = summarize_tool()

        return f"Tool Used: Summarize Tool\n\n{answer}"

    else:

        answer, sources = qa_tool(query)

        return f"Tool Used: Q&A Tool\n\n{answer}\n\nSources: {', '.join(sources)}"

In [ ]:
def process_pdf(file):
    try:
        global pages, chunks, embeddings, full_document, collection

        # Get file path from Gradio
        file_path = file.name if hasattr(file, "name") else file

        pages = load_pdf(file_path)

        chunks = chunk_text(pages)

        embeddings = create_embeddings(chunks)

        import chromadb
        chroma_client = chromadb.Client()

        collection = chroma_client.create_collection(name="document_store")

        for i, chunk in enumerate(chunks):

            collection.add(
                ids=[str(i)],
                documents=[chunk["text"]],
                embeddings=[embeddings[i].tolist()],
                metadatas=[{"page": chunk["page"]}]
            )

        full_document = " ".join([p["text"] for p in pages])

        return "✅ Document processed successfully!"

    except Exception as e:
        return f"❌ Error: {str(e)}"

IMPROVING UI


In [ ]:
import gradio as gr
import chromadb


def process_pdf(file):

    try:
        global pages, chunks, embeddings, full_document, collection

        # Get file path
        file_path = file.name if hasattr(file, "name") else file

        # Load PDF
        pages = load_pdf(file_path)

        # Chunk text
        chunks = chunk_text(pages)

        # Create embeddings
        embeddings = create_embeddings(chunks)

        # Initialize ChromaDB
        chroma_client = chromadb.Client()

        # Get or create collection
        collection = chroma_client.get_or_create_collection(name="document_store")

        # Clear previous embeddings
        try:
            collection.delete(where={})
        except:
            pass

        # Add embeddings
        for i, chunk in enumerate(chunks):

            collection.add(
                ids=[str(i)],
                documents=[chunk["text"]],
                embeddings=[embeddings[i].tolist()],
                metadatas=[{"page": chunk["page"]}]
            )

        # Store full document for summarization
        full_document = " ".join([p["text"] for p in pages])

        return "✅ Document processed successfully!"

    except Exception as e:
        return f"❌ Error: {str(e)}"


def ask_question(query):

    try:

        intent = classify_intent(query)

        if "summarize" in intent:

            answer = summarize_tool()

            return f"Tool Used: Summarize Tool\n\n{answer}"

        else:

            answer, sources = qa_tool(query)

            return f"Tool Used: Q&A Tool\n\n{answer}\n\nSources: {', '.join(sources)}"

    except Exception as e:

        return f"Error: {str(e)}"


with gr.Blocks() as demo:

    gr.Markdown("# 📄 AI Document Assistant")

    file = gr.File(label="Upload PDF", file_types=[".pdf"])

    process_btn = gr.Button("Process Document")

    status = gr.Textbox(
        label="Status",
        lines=3
    )

    process_btn.click(
        fn=process_pdf,
        inputs=file,
        outputs=status
    )

    query = gr.Textbox(
        label="Ask a question",
        lines=3,
        placeholder="Type your question here..."
    )

    answer = gr.Textbox(
        label="AI Response",
        lines=20,      # BIGGER RESPONSE AREA
        max_lines=40,  # Allows long responses
        show_copy_button=True
    )

    query.submit(
        fn=ask_question,
        inputs=query,
        outputs=answer
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aad5c8b7c359912632.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ADDING DWNLD OPTION

In [ ]:
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=2e829cfb6b6511aca4c62ee3a38a49448c827b22ebfcd170e7604cda938df1be
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [ ]:
from fpdf import FPDF
import gradio as gr
import chromadb


# ✅ PDF FUNCTION


def export_response_to_pdf(text):

    if not text or text.strip() == "":
        return None

    # ✅ CLEAN TEXT (important fix)
    clean_text = (
        text.replace("**", "")   # remove markdown bold
            .replace("’", "'")  # fix unicode quotes
            .replace("“", '"')
            .replace("”", '"')
    )

    # remove any unsupported characters
    clean_text = clean_text.encode("latin-1", "ignore").decode("latin-1")

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    for line in clean_text.split("\n"):
        pdf.multi_cell(0, 8, line)

    file_path = "ai_response.pdf"
    pdf.output(file_path)

    return file_path


def process_pdf(file):

    try:
        global pages, chunks, embeddings, full_document, collection

        file_path = file.name if hasattr(file, "name") else file

        pages = load_pdf(file_path)
        chunks = chunk_text(pages)
        embeddings = create_embeddings(chunks)

        chroma_client = chromadb.Client()
        collection = chroma_client.get_or_create_collection(name="document_store")

        try:
            collection.delete(where={})
        except:
            pass

        for i, chunk in enumerate(chunks):
            collection.add(
                ids=[str(i)],
                documents=[chunk["text"]],
                embeddings=[embeddings[i].tolist()],
                metadatas=[{"page": chunk["page"]}]
            )

        full_document = " ".join([p["text"] for p in pages])

        return "✅ Document processed successfully!"

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ✅ UPDATED FUNCTION (returns 2 outputs now)
def ask_question(query):

    try:
        intent = classify_intent(query)

        if "summarize" in intent:
            answer = summarize_tool()
            output = f"Tool Used: Summarize Tool\n\n{answer}"
        else:
            answer, sources = qa_tool(query)
            output = f"Tool Used: Q&A Tool\n\n{answer}\n\nSources: {', '.join(sources)}"

        return output, output   # 👈 IMPORTANT

    except Exception as e:
        return f"Error: {str(e)}", ""


with gr.Blocks() as demo:

    gr.Markdown("# 📄 AI Document Assistant")

    file = gr.File(label="Upload PDF", file_types=[".pdf"])
    process_btn = gr.Button("Process Document")

    status = gr.Textbox(label="Status", lines=3)

    process_btn.click(
        fn=process_pdf,
        inputs=file,
        outputs=status
    )

    query = gr.Textbox(
        label="Ask a question",
        lines=3,
        placeholder="Type your question here..."
    )

    answer = gr.Textbox(
        label="AI Response",
        lines=20,
        max_lines=40,
        show_copy_button=True
    )

    # ✅ STATE to store latest response
    response_state = gr.State("")

    # ✅ FILE output
    download_file = gr.File(label="Download Response")

    query.submit(
        fn=ask_question,
        inputs=query,
        outputs=[answer, response_state]   # 👈 store response
    )

    # ✅ DOWNLOAD BUTTON
    download_btn = gr.Button("📥 Download as PDF")

    download_btn.click(
        fn=export_response_to_pdf,
        inputs=response_state,   # 👈 uses stored text
        outputs=download_file
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://80296c621b8938208c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://aad5c8b7c359912632.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://80296c621b8938208c.gradio.live
